# von Mises-Fisher Mixture Model on the Sphere

This notebook demonstrates **soft clustering on the sphere** using a mixture of **von Mises-Fisher (vMF)** distributions, trained via the **Expectation-Maximization (EM)** algorithm.

## Why Not Just Use a Gaussian on a Sphere?

A Gaussian distribution lives in flat (Euclidean) space — it assigns probability to points spreading outward in all directions from the mean. But on a sphere, there is no "outward" — every direction curves back. A Gaussian centered at the North Pole would leak probability *through* the sphere, assigning mass to points that don't even exist on the surface.

The **von Mises-Fisher distribution** is the natural "Gaussian of the sphere." Just as a Gaussian is the maximum-entropy distribution on $\mathbb{R}^d$ with a given mean and variance, the vMF is the maximum-entropy distribution on the sphere $S^{p-1}$ with a given mean direction.

## From Spherical K-Means to vMF Mixture

In the companion notebook (`spherical_kmeans.ipynb`), we clustered points on a sphere using Haversine distance with **hard** assignments. The vMF mixture model extends this with:
- **Soft assignments:** Each point gets a probability of belonging to each cluster
- **Variable concentration:** Each cluster can be tight or spread out (different $\kappa$ values)
- **Proper density model:** We can compute the actual likelihood of the data

**Use cases:** Directional data (wind directions, animal migration), text clustering (TF-IDF vectors on the unit sphere), geographic data, astronomical observations.

## 1. Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
from scipy.special import ive  # exponentially scaled Bessel function

np.random.seed(42)

print("Setup complete!")

## 2. Generate Pre-Clustered Points on the Unit Sphere

We generate 5 clusters on the unit sphere using the vMF distribution itself, with **deliberately different concentrations** $\kappa$:
- One very tight cluster ($\kappa = 80$) — points bunched closely
- One very spread cluster ($\kappa = 10$) — points scattered widely
- Three medium clusters ($\kappa = 30\text{--}50$)

This is where the vMF mixture shines over spherical K-Means: K-Means treats all clusters as having the same spread, but our data doesn't.

In [ ]:
def sample_vmf(mu, kappa, n, rng):
    """
    Sample n points from vMF(mu, kappa) on the unit sphere S^2 in R^3.
    
    Uses the Wood (1994) algorithm:
    1. Sample the "height" w along the mean direction from a special 1D distribution
    2. Sample a uniform direction in the tangent plane
    3. Combine to get a point on the sphere
    4. Rotate so the mean direction aligns with mu
    """
    p = 3  # dimension of ambient space
    mu = np.asarray(mu, dtype=np.float64)
    mu = mu / np.linalg.norm(mu)  # ensure unit vector
    
    # Step 1: Sample w (the cosine of the angle from the mean direction)
    # Using rejection sampling with the envelope from Wood (1994)
    b = (p - 1) / (2 * kappa + np.sqrt(4 * kappa**2 + (p - 1)**2))
    x0 = (1 - b) / (1 + b)
    c = kappa * x0 + (p - 1) * np.log(1 - x0**2)
    
    samples = []
    while len(samples) < n:
        # Over-sample to reduce loop iterations
        batch = max(n * 3, 100)
        z = rng.beta((p - 1) / 2, (p - 1) / 2, size=batch)
        w_candidates = (1 - (1 + b) * z) / (1 - (1 - b) * z)
        u = rng.uniform(0, 1, size=batch)
        
        accept = kappa * w_candidates + (p - 1) * np.log(1 - x0 * w_candidates) - c
        accepted = w_candidates[np.log(u) < accept]
        samples.extend(accepted.tolist())
    
    w = np.array(samples[:n])  # cosine of angle from mean
    
    # Step 2: Sample uniform direction in tangent plane
    v = rng.normal(size=(n, p - 1))
    v = v / np.linalg.norm(v, axis=1, keepdims=True)
    
    # Step 3: Combine — points around the North Pole [0, 0, 1]
    r = np.sqrt(1 - w**2)  # sin of angle
    points_np = np.zeros((n, p))
    points_np[:, 0] = r * v[:, 0]
    points_np[:, 1] = r * v[:, 1]
    points_np[:, 2] = w
    
    # Step 4: Rotate from North Pole to mu
    north = np.array([0.0, 0.0, 1.0])
    if np.allclose(mu, north):
        return points_np
    elif np.allclose(mu, -north):
        points_np[:, 2] *= -1
        return points_np
    
    # Rotation via Rodrigues' formula
    axis = np.cross(north, mu)
    axis = axis / np.linalg.norm(axis)
    cos_angle = np.dot(north, mu)
    sin_angle = np.sqrt(1 - cos_angle**2)
    
    # Rotation matrix
    K_mat = np.array([[0, -axis[2], axis[1]],
                      [axis[2], 0, -axis[0]],
                      [-axis[1], axis[0], 0]])
    R = np.eye(3) + sin_angle * K_mat + (1 - cos_angle) * K_mat @ K_mat
    
    return (R @ points_np.T).T


def generate_vmf_clusters(n_points=300, seed=42):
    """
    Generate clusters on the unit sphere with different concentrations.
    Returns points (N,3), true labels, true means, true kappas.
    """
    rng = np.random.default_rng(seed)
    
    # Cluster configs: (mean_direction as lat/lon in degrees, kappa, n_points)
    configs = [
        {'mean_latlon': (45, 30),   'kappa': 50, 'n': 60},   # medium-tight, northern
        {'mean_latlon': (10, 150),  'kappa': 10, 'n': 70},   # very spread, equatorial
        {'mean_latlon': (-30, 260), 'kappa': 80, 'n': 50},   # very tight, southern
        {'mean_latlon': (60, -60),  'kappa': 30, 'n': 60},   # medium, high latitude
        {'mean_latlon': (-50, 90),  'kappa': 40, 'n': 60},   # medium, southern
    ]
    
    all_points = []
    all_labels = []
    true_means = []
    true_kappas = []
    
    for i, cfg in enumerate(configs):
        lat, lon = cfg['mean_latlon']
        lat_r, lon_r = np.radians(lat), np.radians(lon)
        # Convert lat/lon to 3D unit vector
        mu = np.array([
            np.cos(lat_r) * np.cos(lon_r),
            np.cos(lat_r) * np.sin(lon_r),
            np.sin(lat_r)
        ])
        
        pts = sample_vmf(mu, cfg['kappa'], cfg['n'], rng)
        all_points.append(pts)
        all_labels.append(np.full(cfg['n'], i))
        true_means.append(mu)
        true_kappas.append(cfg['kappa'])
    
    points = np.vstack(all_points)
    labels = np.concatenate(all_labels)
    
    return points, labels, np.array(true_means), np.array(true_kappas)


points, true_labels, true_means, true_kappas = generate_vmf_clusters()
K = len(true_means)

print(f"Generated {len(points)} points in {K} clusters on S^2")
print(f"Cluster sizes: {[np.sum(true_labels == i) for i in range(K)]}")
print(f"True concentrations (kappa): {true_kappas.tolist()}")
print(f"All points on unit sphere: {np.allclose(np.linalg.norm(points, axis=1), 1.0)}")

## 3. The von Mises-Fisher Distribution

The vMF distribution on the unit sphere $S^{p-1}$ in $\mathbb{R}^p$ is:

$$f(x \mid \mu, \kappa) = C_p(\kappa) \, \exp\!\left(\kappa \, \mu^T x\right)$$

where:

| Symbol | Name | Meaning |
|--------|------|---------|
| $x \in S^{p-1}$ | Observation | A unit vector on the sphere |
| $\mu \in S^{p-1}$ | Mean direction | The "center" of the distribution (unit vector) |
| $\kappa \geq 0$ | Concentration | Controls tightness (like $1/\sigma^2$ for Gaussians) |
| $C_p(\kappa)$ | Normalizing constant | Ensures the density integrates to 1 over the sphere |

The normalizing constant is:

$$C_p(\kappa) = \frac{\kappa^{p/2 - 1}}{(2\pi)^{p/2} \, I_{p/2-1}(\kappa)}$$

where $I_\nu(\kappa)$ is the **modified Bessel function of the first kind**.

### For our case ($p = 3$, sphere in 3D):

$$C_3(\kappa) = \frac{\kappa}{4\pi \sinh(\kappa)}$$

### Intuition

- The density depends only on $\mu^T x = \cos(\angle(\mu, x))$ — the **cosine of the angle** between $x$ and the mean direction. Points at the same angular distance from $\mu$ have the same density.
- **$\kappa = 0$:** Uniform distribution on the sphere (no preferred direction)
- **$\kappa \to \infty$:** All mass concentrated at $\mu$ (delta function)
- **$\kappa \approx 1$:** Weakly concentrated (broad cap)
- **$\kappa \approx 50$:** Tightly concentrated (small cap)

In [ ]:
def log_vmf_normalizing_constant(kappa, p=3):
    """
    Compute log C_p(kappa) using the exponentially-scaled Bessel function
    for numerical stability.
    
    C_p(kappa) = kappa^(p/2-1) / ((2*pi)^(p/2) * I_{p/2-1}(kappa))
    
    We use ive(v, kappa) = I_v(kappa) * exp(-kappa) to avoid overflow.
    """
    v = p / 2.0 - 1  # order of Bessel function
    
    # log I_v(kappa) = log(ive(v, kappa)) + kappa
    log_ive = np.log(np.clip(ive(v, kappa), 1e-300, None))
    log_bessel = log_ive + kappa
    
    log_C = (v * np.log(np.clip(kappa, 1e-300, None))
             - (p / 2.0) * np.log(2 * np.pi)
             - log_bessel)
    
    return log_C


def log_vmf_pdf(x, mu, kappa, p=3):
    """
    Compute log f(x | mu, kappa) for the vMF distribution.
    
    x: (N, p) array of unit vectors
    mu: (p,) mean direction (unit vector)
    kappa: concentration parameter
    
    Returns: (N,) array of log-densities
    """
    log_C = log_vmf_normalizing_constant(kappa, p)
    cos_angle = x @ mu  # (N,) — dot product = cos(angle)
    return log_C + kappa * cos_angle


# Sanity checks
print("von Mises-Fisher sanity checks:")
print(f"  C_3(kappa=1): log = {log_vmf_normalizing_constant(1.0):.4f}, "
      f"C = {np.exp(log_vmf_normalizing_constant(1.0)):.4f}")
print(f"  C_3(kappa=10): log = {log_vmf_normalizing_constant(10.0):.4f}")
print(f"  C_3(kappa=100): log = {log_vmf_normalizing_constant(100.0):.4f}")

# For p=3: C_3(kappa) = kappa / (4*pi*sinh(kappa))
for k in [1.0, 10.0, 50.0]:
    exact = k / (4 * np.pi * np.sinh(k))
    computed = np.exp(log_vmf_normalizing_constant(k))
    print(f"  kappa={k:.0f}: exact C_3={exact:.6e}, computed={computed:.6e}, "
          f"match={np.isclose(exact, computed)}")

In [ ]:
# Visualize vMF density for different kappa values on the sphere
fig, axes = plt.subplots(1, 4, figsize=(22, 5),
                          subplot_kw={'projection': '3d'})

mu_demo = np.array([0, 0, 1.0])  # North Pole
kappas_demo = [1, 10, 50, 200]

# Create sphere grid
u_s = np.linspace(0, 2 * np.pi, 80)
v_s = np.linspace(0, np.pi, 40)
x_s = np.outer(np.cos(u_s), np.sin(v_s))
y_s = np.outer(np.sin(u_s), np.sin(v_s))
z_s = np.outer(np.ones_like(u_s), np.cos(v_s))

for ax, kappa_d in zip(axes, kappas_demo):
    # Compute density on sphere surface
    sphere_pts = np.stack([x_s.ravel(), y_s.ravel(), z_s.ravel()], axis=-1)
    log_dens = log_vmf_pdf(sphere_pts, mu_demo, kappa_d)
    dens = np.exp(log_dens - log_dens.max())  # normalize for coloring
    dens_grid = dens.reshape(x_s.shape)
    
    # Color the sphere by density
    cmap = plt.cm.YlOrRd
    colors = cmap(dens_grid)
    ax.plot_surface(x_s, y_s, z_s, facecolors=colors,
                    alpha=0.8, linewidth=0, antialiased=True)
    
    ax.set_xlim([-1.3, 1.3])
    ax.set_ylim([-1.3, 1.3])
    ax.set_zlim([-1.3, 1.3])
    ax.set_box_aspect([1, 1, 1])
    ax.set_title(f'\u03ba = {kappa_d}', fontsize=13, fontweight='bold')
    ax.set_xlabel('X', fontsize=8)
    ax.set_ylabel('Y', fontsize=8)
    ax.set_zlabel('Z', fontsize=8)

plt.suptitle('vMF Density on the Sphere (mean = North Pole)\n'
             'Low \u03ba = spread out, High \u03ba = concentrated',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. EM Algorithm for vMF Mixture

A vMF mixture models the data as:

$$p(x) = \sum_{k=1}^K \pi_k \, f(x \mid \mu_k, \kappa_k)$$

### E-step: Compute Responsibilities

$$r_{nk} = \frac{\pi_k \, C_p(\kappa_k) \, \exp(\kappa_k \, \mu_k^T x_n)}{\sum_{j=1}^K \pi_j \, C_p(\kappa_j) \, \exp(\kappa_j \, \mu_j^T x_n)}$$

This is identical in form to the Gaussian case — just swap the density function.

### M-step: Update Parameters

**Mean direction:** Compute the weighted resultant vector, then normalize:

$$\bar{r}_k = \sum_{n=1}^N r_{nk} \, x_n, \qquad \mu_k = \frac{\bar{r}_k}{\|\bar{r}_k\|}$$

**Concentration $\kappa$:** There's no closed-form solution. We use the approximation from Banerjee et al. (2005) based on the **mean resultant length** $\bar{R}_k$:

$$N_k = \sum_{n} r_{nk}, \qquad \bar{R}_k = \frac{\|\bar{r}_k\|}{N_k}$$

$$\kappa_k \approx \frac{\bar{R}_k \, (p - \bar{R}_k^2)}{1 - \bar{R}_k^2}$$

This approximation is accurate for moderate to high $\kappa$ and works well in practice.

**Mixing weights:**

$$\pi_k = \frac{N_k}{N}$$

### Log-Likelihood

$$\mathcal{L} = \sum_{n=1}^N \log \sum_{k=1}^K \pi_k \, f(x_n \mid \mu_k, \kappa_k)$$

In [ ]:
def log_sum_exp(log_vals, axis=None):
    """Numerically stable log-sum-exp."""
    max_val = np.max(log_vals, axis=axis, keepdims=True)
    return np.squeeze(max_val, axis=axis) + np.log(
        np.sum(np.exp(log_vals - max_val), axis=axis)
    )


def estimate_kappa(R_bar, p=3):
    """
    Estimate kappa from mean resultant length R_bar.
    Banerjee et al. (2005) approximation.
    """
    R_bar = np.clip(R_bar, 1e-10, 1 - 1e-10)
    return R_bar * (p - R_bar**2) / (1 - R_bar**2)


def em_vmf_mixture(points, k, max_iter=200, tol=1e-6, seed=123):
    """
    EM algorithm for a mixture of von Mises-Fisher distributions.
    
    Parameters:
    -----------
    points : (N, 3) array of unit vectors on S^2
    k : number of components
    max_iter : maximum iterations
    tol : convergence threshold on log-likelihood change
    seed : random seed
    
    Returns:
    --------
    labels, responsibilities, means, kappas, weights, ll_history
    """
    rng = np.random.default_rng(seed)
    N, p = points.shape
    
    # --- Initialization ---
    init_idx = rng.choice(N, size=k, replace=False)
    means = points[init_idx].copy()
    means = means / np.linalg.norm(means, axis=1, keepdims=True)
    
    kappas = np.full(k, 20.0)  # initial guess
    weights = np.ones(k) / k
    
    ll_history = []
    
    for iteration in range(max_iter):
        # === E-step ===
        log_resp = np.zeros((N, k))
        for j in range(k):
            log_resp[:, j] = np.log(weights[j]) + log_vmf_pdf(
                points, means[j], kappas[j], p
            )
        
        # Normalize
        log_resp_norm = log_sum_exp(log_resp, axis=1)  # (N,)
        log_resp_normalized = log_resp - log_resp_norm[:, np.newaxis]
        responsibilities = np.exp(log_resp_normalized)
        
        # Log-likelihood
        ll = np.sum(log_resp_norm)
        ll_history.append(ll)
        
        if iteration > 0 and abs(ll - ll_history[-2]) < tol:
            print(f"Converged at iteration {iteration + 1}")
            break
        
        # === M-step ===
        for j in range(k):
            r_j = responsibilities[:, j]  # (N,)
            N_j = np.sum(r_j)
            
            if N_j < 1e-10:
                # Dead component — reinitialize
                means[j] = points[rng.choice(N)]
                kappas[j] = 20.0
                weights[j] = 1.0 / k
                continue
            
            # Weighted resultant vector
            r_bar_vec = np.sum(r_j[:, np.newaxis] * points, axis=0)
            r_bar_norm = np.linalg.norm(r_bar_vec)
            
            # Updated mean direction
            means[j] = r_bar_vec / max(r_bar_norm, 1e-10)
            
            # Mean resultant length
            R_bar = r_bar_norm / N_j
            
            # Updated kappa
            kappas[j] = estimate_kappa(R_bar, p)
            kappas[j] = np.clip(kappas[j], 0.1, 1000)  # safety bounds
            
            # Updated weight
            weights[j] = N_j / N
    else:
        print(f"Reached max iterations ({max_iter})")
    
    labels = np.argmax(responsibilities, axis=1)
    
    return labels, responsibilities, means, kappas, weights, ll_history


# Run EM
vmf_labels, vmf_resp, vmf_means, vmf_kappas, vmf_weights, vmf_ll = em_vmf_mixture(
    points, k=K, max_iter=200
)

print(f"\nCluster sizes: {[np.sum(vmf_labels == i) for i in range(K)]}")
print(f"Mixing weights: {[f'{w:.3f}' for w in vmf_weights]}")
print(f"Estimated kappas: {[f'{k:.1f}' for k in vmf_kappas]}")
print(f"True kappas:      {true_kappas.tolist()}")
print(f"Final log-likelihood: {vmf_ll[-1]:.2f}")

## 5. Visualizations

### 5a. 3D Sphere with Clustered Points

Points on the unit sphere colored by vMF mixture assignment. The mean directions are shown as large black stars.

In [ ]:
cluster_colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Draw translucent sphere wireframe
u_w = np.linspace(0, 2 * np.pi, 30)
v_w = np.linspace(0, np.pi, 20)
x_w = np.outer(np.cos(u_w), np.sin(v_w))
y_w = np.outer(np.sin(u_w), np.sin(v_w))
z_w = np.outer(np.ones_like(u_w), np.cos(v_w))
ax.plot_wireframe(x_w, y_w, z_w, alpha=0.05, color='gray', linewidth=0.5)

# Plot clustered points
for i in range(K):
    mask = vmf_labels == i
    ax.scatter(points[mask, 0], points[mask, 1], points[mask, 2],
               c=cluster_colors[i], s=25, alpha=0.7,
               label=f'Cluster {i+1} (\u03ba\u2248{vmf_kappas[i]:.0f})',
               depthshade=True, zorder=5)

# Mean directions as stars (slightly outside sphere for visibility)
scale = 1.05
ax.scatter(vmf_means[:, 0] * scale, vmf_means[:, 1] * scale,
           vmf_means[:, 2] * scale,
           c='black', marker='*', s=300, edgecolors='white',
           linewidths=1.5, zorder=10, label='Mean directions')

ax.set_xlim([-1.3, 1.3])
ax.set_ylim([-1.3, 1.3])
ax.set_zlim([-1.3, 1.3])
ax.set_box_aspect([1, 1, 1])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('vMF Mixture Clustering on the Unit Sphere',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

### 5b. Latitude-Longitude Map Projection

The sphere unwrapped into a 2D map. Background shading shows the mixture density — notice how clusters with lower $\kappa$ produce broader regions.

In [ ]:
def cart_to_latlon(xyz):
    """Convert (N,3) Cartesian unit vectors to (lat, lon) in degrees."""
    lat = np.degrees(np.arcsin(np.clip(xyz[:, 2], -1, 1)))
    lon = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
    return lat, lon


def latlon_to_cart(lat_deg, lon_deg):
    """Convert lat/lon in degrees to (N,3) unit vectors."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    return np.stack([x, y, z], axis=-1)


lat, lon = cart_to_latlon(points)
mean_lat, mean_lon = cart_to_latlon(vmf_means)

fig, ax = plt.subplots(figsize=(14, 7))

# Background density shading
grid_res = 100
g_lon = np.linspace(-180, 180, grid_res)
g_lat = np.linspace(-90, 90, grid_res // 2)
g_lon_mesh, g_lat_mesh = np.meshgrid(g_lon, g_lat)
grid_xyz = latlon_to_cart(g_lat_mesh.ravel(), g_lon_mesh.ravel())

# Compute posterior for grid
log_resp_grid = np.zeros((len(grid_xyz), K))
for j in range(K):
    log_resp_grid[:, j] = np.log(vmf_weights[j]) + log_vmf_pdf(
        grid_xyz, vmf_means[j], vmf_kappas[j]
    )
grid_labels = np.argmax(log_resp_grid, axis=1)
grid_img = grid_labels.reshape(grid_res // 2, grid_res)

bg_rgba = np.ones((grid_res // 2, grid_res, 4))
for i in range(K):
    mask_2d = grid_img == i
    bg_rgba[mask_2d] = mcolors.to_rgba(cluster_colors[i], alpha=0.15)

ax.imshow(bg_rgba, extent=[-180, 180, -90, 90],
          origin='lower', aspect='auto')

# Plot points
for i in range(K):
    mask = vmf_labels == i
    ax.scatter(lon[mask], lat[mask],
               c=cluster_colors[i], s=25, alpha=0.8, edgecolors='white',
               linewidths=0.3, label=f'Cluster {i+1}', zorder=5)

# Mean directions
ax.scatter(mean_lon, mean_lat,
           c='black', marker='*', s=300, edgecolors='white',
           linewidths=1.5, zorder=10, label='Mean directions')

ax.set_xlabel('Longitude (\u00b0)', fontsize=12)
ax.set_ylabel('Latitude (\u00b0)', fontsize=12)
ax.set_title('vMF Mixture — Latitude/Longitude Projection\n'
             '(background = Voronoi regions by posterior probability)',
             fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 5c. Soft Assignment Visualization

Points deep in a cluster have high confidence (opaque); points between clusters are uncertain (translucent). With different $\kappa$ values, the tight cluster has very confident assignments while the spread cluster has more uncertainty at its edges.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: opacity = confidence
ax = axes[0]
max_resp = np.max(vmf_resp, axis=1)
for i in range(K):
    mask = vmf_labels == i
    rgba = mcolors.to_rgba(cluster_colors[i])
    colors_arr = np.tile(rgba, (np.sum(mask), 1))
    colors_arr[:, 3] = max_resp[mask]
    ax.scatter(lon[mask], lat[mask],
               c=colors_arr, s=30, edgecolors='white',
               linewidths=0.3, zorder=5)

ax.scatter(mean_lon, mean_lat,
           c='black', marker='*', s=300, edgecolors='white',
           linewidths=1.5, zorder=10)
ax.set_xlabel('Longitude (\u00b0)', fontsize=12)
ax.set_ylabel('Latitude (\u00b0)', fontsize=12)
ax.set_title('Soft Assignment: Opacity = Confidence\n'
             '(faint = uncertain, solid = confident)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)

# Right: histogram of max responsibilities
ax = axes[1]
ax.hist(max_resp, bins=50, color='#2c3e50', edgecolor='white', alpha=0.8)
ax.axvline(1.0 / K, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Random guess = {1/K:.2f}')
ax.set_xlabel('Maximum Responsibility', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Assignment Confidence',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.15)

plt.suptitle('vMF Mixture Soft Assignments',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5d. Convergence Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

iterations = range(1, len(vmf_ll) + 1)

ax.plot(iterations, vmf_ll, 'o-', color='#2c3e50', markersize=6,
        linewidth=2, markerfacecolor='#9b59b6', markeredgecolor='white',
        markeredgewidth=1.5)

ax.fill_between(iterations, vmf_ll, alpha=0.1, color='#9b59b6')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Log-Likelihood', fontsize=12)
ax.set_title('vMF Mixture EM Convergence',
             fontsize=14, fontweight='bold')

ax.annotate(f'Start: {vmf_ll[0]:.1f}', xy=(1, vmf_ll[0]),
            xytext=(3, vmf_ll[0]),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')
ax.annotate(f'Final: {vmf_ll[-1]:.1f}', xy=(len(vmf_ll), vmf_ll[-1]),
            xytext=(max(1, len(vmf_ll) - 5), vmf_ll[-1] - (vmf_ll[-1] - vmf_ll[0]) * 0.1),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5e. Concentration Comparison: True vs Estimated $\kappa$

Did EM recover the true concentration parameters? We compare estimated $\kappa$ values with the ground truth. Since cluster label ordering may differ, we match by closest mean direction.

In [ ]:
# Match estimated clusters to true clusters by closest mean direction
# (cosine similarity between estimated and true means)
cos_sim = vmf_means @ true_means.T  # (K, K)
matched_order = []
used = set()
for i in range(K):
    # For each estimated mean, find the closest true mean (not already matched)
    sims = cos_sim[i].copy()
    for u in used:
        sims[u] = -2  # exclude
    best = np.argmax(sims)
    matched_order.append(best)
    used.add(best)

print("Cluster matching (estimated -> true):")
for i, j in enumerate(matched_order):
    cos_val = cos_sim[i, j]
    print(f"  Est. cluster {i} -> True cluster {j}: "
          f"cos(angle) = {cos_val:.4f}, "
          f"kappa est = {vmf_kappas[i]:.1f}, true = {true_kappas[j]:.0f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))

x_pos = np.arange(K)
width = 0.35

true_k_matched = [true_kappas[j] for j in matched_order]
est_k = vmf_kappas.tolist()

bars1 = ax.bar(x_pos - width/2, true_k_matched, width,
               label='True \u03ba', color='#3498db', alpha=0.8, edgecolor='white')
bars2 = ax.bar(x_pos + width/2, est_k, width,
               label='Estimated \u03ba', color='#e74c3c', alpha=0.8, edgecolor='white')

# Value labels
for bar, val in zip(bars1, true_k_matched):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', fontsize=11, fontweight='bold', color='#3498db')
for bar, val in zip(bars2, est_k):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', fontsize=11, fontweight='bold', color='#e74c3c')

ax.set_xlabel('Cluster', fontsize=12)
ax.set_ylabel('Concentration (\u03ba)', fontsize=12)
ax.set_title('True vs Estimated Concentration Parameters',
             fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Cluster {i+1}' for i in range(K)])
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Spherical K-Means vs vMF Mixture — Head-to-Head

Spherical K-Means treats all clusters as having the **same spread** and gives **hard** assignments. The vMF mixture can model clusters with different concentrations and gives **soft** assignments. Let's see where this matters.

In [ ]:
def haversine(u, v):
    """Great-circle distance between 3D unit vectors."""
    cos_angle = np.clip(np.sum(u * v, axis=-1), -1, 1)
    return np.arccos(cos_angle)


def spherical_kmeans(points, k=5, max_iter=100, tol=1e-6, seed=123):
    """Spherical K-Means using great-circle distance and spherical means."""
    rng = np.random.default_rng(seed)
    n = len(points)
    
    init_idx = rng.choice(n, size=k, replace=False)
    centroids = points[init_idx].copy()
    centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    
    history = []
    labels = np.zeros(n, dtype=int)
    
    for iteration in range(max_iter):
        # Assignment: great-circle distance
        dist_matrix = np.zeros((n, k))
        for j in range(k):
            dist_matrix[:, j] = haversine(points, centroids[j])
        labels = np.argmin(dist_matrix, axis=1)
        
        total_dist = np.sum(dist_matrix[np.arange(n), labels])
        history.append(total_dist)
        
        # Update: spherical mean (normalize the Euclidean mean)
        new_centroids = np.zeros_like(centroids)
        for j in range(k):
            members = points[labels == j]
            if len(members) == 0:
                new_centroids[j] = points[rng.choice(n)]
            else:
                mean_vec = members.mean(axis=0)
                norm = np.linalg.norm(mean_vec)
                new_centroids[j] = mean_vec / max(norm, 1e-10)
        
        shift = np.max(haversine(centroids, new_centroids))
        centroids = new_centroids
        
        if shift < tol:
            print(f"Spherical K-Means converged at iteration {iteration + 1}")
            break
    else:
        print(f"Spherical K-Means reached max iterations ({max_iter})")
    
    return labels, centroids, history


skm_labels, skm_centroids, skm_history = spherical_kmeans(points, k=K)

print(f"Spherical K-Means sizes: {[np.sum(skm_labels == i) for i in range(K)]}")
print(f"vMF Mixture sizes:       {[np.sum(vmf_labels == i) for i in range(K)]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

skm_lat, skm_lon = cart_to_latlon(skm_centroids)

configs = [
    ('Spherical K-Means (Haversine)', skm_labels, skm_centroids),
    ('vMF Mixture (EM)', vmf_labels, vmf_means)
]

for ax, (title, lbls, cents) in zip(axes, configs):
    # Background: Voronoi regions
    if 'K-Means' in title:
        grid_dists = np.zeros((len(grid_xyz), K))
        for j in range(K):
            grid_dists[:, j] = haversine(grid_xyz, cents[j])
        bg_labels = np.argmin(grid_dists, axis=1)
    else:
        log_r = np.zeros((len(grid_xyz), K))
        for j in range(K):
            log_r[:, j] = np.log(vmf_weights[j]) + log_vmf_pdf(
                grid_xyz, cents[j], vmf_kappas[j]
            )
        bg_labels = np.argmax(log_r, axis=1)
    
    bg_img = bg_labels.reshape(grid_res // 2, grid_res)
    bg = np.ones((grid_res // 2, grid_res, 4))
    for i in range(K):
        mask_2d = bg_img == i
        bg[mask_2d] = mcolors.to_rgba(cluster_colors[i], alpha=0.15)
    ax.imshow(bg, extent=[-180, 180, -90, 90], origin='lower', aspect='auto')
    
    c_lat, c_lon = cart_to_latlon(cents)
    
    for i in range(K):
        mask = lbls == i
        ax.scatter(lon[mask], lat[mask],
                   c=cluster_colors[i], s=25, alpha=0.8, edgecolors='white',
                   linewidths=0.3, label=f'Cluster {i+1}', zorder=5)
    
    ax.scatter(c_lon, c_lat,
               c='black', marker='*', s=300, edgecolors='white',
               linewidths=1.5, zorder=10, label='Centers')
    
    ax.set_xlabel('Longitude (\u00b0)', fontsize=12)
    ax.set_ylabel('Latitude (\u00b0)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Same Data: Spherical K-Means (Equal Spread) vs vMF Mixture (Variable Spread)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6b. Agreement Analysis

In [ ]:
from itertools import combinations


def rand_index(labels_a, labels_b):
    """Compute the Rand Index between two clusterings."""
    n = len(labels_a)
    agree = 0
    total = 0
    for i, j in combinations(range(n), 2):
        same_a = labels_a[i] == labels_a[j]
        same_b = labels_b[i] == labels_b[j]
        if same_a == same_b:
            agree += 1
        total += 1
    return agree / total if total > 0 else 1.0


ri_skm_true = rand_index(skm_labels, true_labels)
ri_vmf_true = rand_index(vmf_labels, true_labels)
ri_skm_vmf = rand_index(skm_labels, vmf_labels)

print("Rand Index (agreement with ground truth):")
print(f"  Spherical K-Means vs True:  {ri_skm_true:.4f}")
print(f"  vMF Mixture (EM)  vs True:  {ri_vmf_true:.4f}")
print(f"  K-Means vs vMF Mixture:     {ri_skm_vmf:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))

methods = ['Spherical K-Means', 'vMF Mixture (EM)']
scores = [ri_skm_true, ri_vmf_true]
bar_colors = ['#3498db', '#e74c3c']

bars = ax.barh(methods, scores, color=bar_colors, edgecolor='white',
               height=0.5, alpha=0.85)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() - 0.02, bar.get_y() + bar.get_height() / 2,
            f'{score:.4f}', va='center', ha='right', fontsize=14,
            fontweight='bold', color='white')

ax.set_xlabel('Rand Index (vs Ground Truth)', fontsize=12)
ax.set_title('Clustering Accuracy: Spherical K-Means vs vMF Mixture',
             fontsize=14, fontweight='bold')
ax.set_xlim(0.7, 1.0)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Choosing K — PVE & Silhouette Analysis

How do we know K = 5 is the right number of clusters? We use two metrics:

**PVE (Proportion of Variance Explained)**

$$\text{PVE}(K) = 1 - \frac{\text{TWCSS}_K}{\text{TWCSS}_1}$$

where $\text{TWCSS}_K$ is the **total within-cluster sum of squared distances** for K clusters and $\text{TWCSS}_1$ is the single-cluster baseline (all points assigned to one cluster). PVE ranges from 0 to 1; an "elbow" in the curve marks a natural K choice.

**Silhouette Score**

For each point $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, +1]$$

where $a(i)$ = mean distance to other points in the **same** cluster (cohesion), and $b(i)$ = mean distance to points in the **nearest other** cluster (separation). Computed from the precomputed pairwise distance matrix so it automatically respects this notebook's metric.

In [ ]:
def silhouette_from_matrix(D, labels):
    """
    Mean silhouette score from a precomputed N×N distance matrix.
    s(i) = (b(i) - a(i)) / max(a(i), b(i))
    """
    n = len(labels)
    unique_clusters = np.unique(labels)
    if len(unique_clusters) <= 1:
        return 0.0
    scores = np.zeros(n)
    for i in range(n):
        ci = labels[i]
        same_mask = (labels == ci).copy()
        same_mask[i] = False
        a_i = float(np.mean(D[i, same_mask])) if np.any(same_mask) else 0.0
        b_i = np.inf
        for cj in unique_clusters:
            if cj == ci:
                continue
            other_mask = (labels == cj)
            if np.any(other_mask):
                b_i = min(b_i, float(np.mean(D[i, other_mask])))
        denom = max(a_i, b_i)
        scores[i] = (b_i - a_i) / denom if denom > 1e-15 else 0.0
    return float(np.mean(scores))

In [ ]:
# For the vMF mixture, use hard assignments (vmf_labels = argmax of vmf_resp)
# and great-circle (haversine) distance for TWCSS and silhouette.
# haversine() working on 3D unit vectors is already defined above.

# Build full N×N great-circle pairwise distance matrix
n_pts = len(points)
print("Building pairwise great-circle distance matrix...")
D_full = np.zeros((n_pts, n_pts))
for i in range(n_pts):
    D_full[i] = haversine(points, points[i])

# TWCSS baseline: one cluster, centred on the spherical mean of all points
overall_center = points.mean(axis=0)
overall_center = overall_center / np.linalg.norm(overall_center)
TWCSS_1 = float(np.sum(haversine(points, overall_center) ** 2))

K_range = range(2, 11)
pve_vals = []
sil_vals = []

print(f"{'K':>4} {'TWCSS':>10} {'PVE':>8} {'Silhouette':>12}")
print("-" * 38)
for k in K_range:
    lbls_k, _, means_k, _, _, _ = em_vmf_mixture(points, k=k, max_iter=100)
    dists_k = haversine(points, means_k[lbls_k])
    twcss_k = float(np.sum(dists_k ** 2))
    pve = 1 - twcss_k / TWCSS_1
    sil = silhouette_from_matrix(D_full, lbls_k)
    pve_vals.append(pve)
    sil_vals.append(sil)
    print(f"{k:>4} {twcss_k:>10.4f} {pve:>8.4f} {sil:>12.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
K_chosen = 5
K_vals = list(K_range)

ax = axes[0]
ax.plot(K_vals, pve_vals, 'o-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#3498db', markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(K_chosen, color='#e74c3c', linestyle='--', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, pve_vals, alpha=0.08, color='#3498db')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('PVE', fontsize=12)
ax.set_title('PVE Elbow Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(K_vals, sil_vals, 's-', color='#2c3e50', linewidth=2, markersize=7,
        markerfacecolor='#9b59b6', markeredgecolor='white', markeredgewidth=1.5)
best_k_sil = K_vals[int(np.argmax(sil_vals))]
ax.axvline(best_k_sil, color='#9b59b6', linestyle='--', linewidth=1.8,
           label=f'Best silhouette K = {best_k_sil}')
ax.axvline(K_chosen, color='#e74c3c', linestyle=':', linewidth=1.8,
           label=f'Chosen K = {K_chosen}')
ax.fill_between(K_vals, sil_vals, alpha=0.08, color='#9b59b6')
ax.set_xlabel('Number of Clusters K', fontsize=12)
ax.set_ylabel('Mean Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_xticks(K_vals); ax.grid(True, alpha=0.3)

plt.suptitle('Choosing K: PVE & Silhouette Validation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nK=2..10 summary:")
print(f"{'K':>4} {'PVE':>8} {'Silhouette':>12}")
print("-" * 27)
for k, pve, sil in zip(K_vals, pve_vals, sil_vals):
    marker = " ◄ chosen" if k == K_chosen else ""
    print(f"{k:>4} {pve:>8.4f} {sil:>12.4f}{marker}")

---
## 7. Summary

| Aspect | Spherical K-Means | vMF Mixture (EM) |
|--------|-------------------|------------------|
| **Assignment** | Hard (0 or 1) | Soft (probabilities $r_{nk}$) |
| **Cluster shape** | Equal-spread spherical caps | Variable-spread caps (different $\kappa_k$) |
| **Parameters per cluster** | Mean direction $\mu_k$ only | Mean $\mu_k$ + Concentration $\kappa_k$ + Weight $\pi_k$ |
| **Handles varying spread** | No — all clusters treated equally | Yes — each cluster gets its own $\kappa$ |
| **Decision boundary** | Great-circle arcs (geodesic Voronoi) | Curved boundaries shaped by $\kappa$ ratios |
| **Distance used** | Haversine (great-circle) | Implicit in vMF density ($\mu^T x$) |
| **Density model** | No (just assignments) | Yes (full probabilistic model) |
| **Best for** | Quick baseline on spherical data | Directional data with varying spread, uncertainty quantification |

### Key Takeaways

1. **Variable concentration is the key advantage.** When clusters on the sphere have genuinely different spreads — one tight, one diffuse — spherical K-Means assigns equal-sized Voronoi cells, misrepresenting the data. The vMF mixture captures this heterogeneity through per-cluster $\kappa$ values.

2. **Soft assignments quantify boundary uncertainty.** A point equidistant from two cluster centers gets a meaningful probability split (e.g., 60/40), rather than an arbitrary hard assignment.

3. **The Banerjee approximation for $\kappa$ works well.** Despite being a simple closed-form approximation, it recovers the true concentration parameters with good accuracy — no iterative Newton-Raphson needed.

4. **The normalizing constant requires care.** The Bessel function $I_\nu(\kappa)$ overflows for large $\kappa$. Using the exponentially-scaled version `ive()` from scipy is essential for numerical stability.

### Next: Hyperbolic — Wrapped Normal Mixture on the Poincar\u00e9 Disk

On the Poincar\u00e9 disk, neither Gaussians nor vMF distributions apply — the space has negative curvature. We'll use a **wrapped normal distribution** adapted to hyperbolic geometry, with the Poincar\u00e9 distance replacing Euclidean distance in the density function.